**Fig. 6 and 7, step 1 (notebook 01): build the shape morphospace (31 features, PCA, Leiden shape clusters, UMAP) that the panels of both figures use**

In [ ]:
# Working directory: data/fig6_7/ under the repository root, where the Fig. 6 and 7 deposit folder is downloaded
# (all_stacks_features.csv, workspace/ = the plugin's Step 4 workspace with the 14 stacks (13 analysed) and their meshes,
# run/ = what these notebooks write and read back, figures/ = the saved panels).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig6_7")
print("working directory:", os.getcwd())

# 01 -- Shape Morphospace

Core shape analysis pipeline: load features, preprocess, PCA, multi-resolution Leiden, UMAP.

**Inputs:** `all_stacks_features.csv`
**Outputs:** `morphospace_results.pkl` (pickle bridge to downstream notebooks)

**Pipeline:**
1. Load merged feature CSV from EpiCycle export
2. Annotate stacks with biological metadata (donor, timepoint, tissue, pattern)
3. Feature selection: shape + size + polarity + skeleton + cross-section
4. Auto log1p for skewed features, Robust scaling, Winsorize
5. Optional: Residual PCA (regress out size PC1)
6. Build KNN graph once, run Leiden at multiple resolutions
7. UMAP embedding (computed once, independent of clustering)
8. Sample representative cells per cluster per resolution
9. Save results pickle for downstream notebooks

In [ ]:
import os, glob, io, json, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.stats import skew
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import NearestNeighbors
from umap import UMAP
import igraph as ig
import leidenalg
import plotly.express as px
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# -- PATHS --
FEATURES_CSV  = r'all_stacks_features.csv'
MESHES_ROOT   = r'workspace/artifacts/step4/meshes/'
OUTPUT_DIR    = r'run'
WORKSPACE_DIR = r'workspace'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -- FEATURE SELECTION --
# NOTE: compactness excluded -- numerically identical to extent (both V/V_bbox).
# NOTE: roughness excluded -- perfect inverse of true_sphericity.
TARGET_FEATURES = [
    'area', 'area_filled', 'equivalent_diameter', 'extent', 'euler_number',
    'bbox_volume', 'fill_ratio',
    'sphericity', 'aspect_ratio', 'z_span', 'z_elongation', 'xy_elongation',
    'ellipsoid_solidity', 'inertia_elongation', 'inertia_flatness',
    'axis_major_length', 'axis_minor_length', 'major_axis_z_angle',
    'surface_area', 'true_sphericity', 'solidity',
    'skeleton_branches', 'skeleton_nodes', 'skeleton_length',
    'apical_area', 'basal_area', 'apical_basal_area_ratio',
    'max_cross_section_z_relative', 'cross_section_taper', 'cross_section_bulge','roughness',
]
SIZE_REGRESSORS = [
    'area', 'area_filled', 'equivalent_diameter', 'bbox_volume',
    'surface_area', 'axis_major_length', 'axis_minor_length',
]

# -- ANALYSIS PARAMS --
N_PCS              = 10
KNN                = 15
LEIDEN_RESOLUTIONS = [0.3, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
ACTIVE_RES         = 1.0
UMAP_NEIGHBORS     = 15
UMAP_MIN_DIST      = 0.1
METRIC             = 'euclidean'
RANDOM_SEED        = 42
WINSOR_CLIP        = 5.0
USE_RESIDUAL_PCA   = False
N_REPS             = 10

# -- VISUAL --
_BASE_PALETTE = [
    '#E74C3C', '#3498DB', '#2ECC71', '#E67E22', '#9B59B6',
    '#F39C12', '#1ABC9C', '#E91E63', '#FF6B6B', '#45B7D1',
    '#A29BFE', '#FD79A8', '#FDCB6E', '#6C5CE7', '#00B894',
    '#D63031', '#0984E3', '#00CEC9', '#E17055', '#74B9FF',
]

def _generate_colors(n):
    """Generate n distinct colors. Base palette first, then HSV cycle."""
    if n <= len(_BASE_PALETTE):
        return _BASE_PALETTE[:n]
    import colorsys
    colors = list(_BASE_PALETTE)
    for i in range(n - len(_BASE_PALETTE)):
        hue = (i * 0.618033988749895) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.7 + (i % 3) * 0.1, 0.85 + (i % 2) * 0.1)
        colors.append('#' + '%02x%02x%02x' % (int(r*255), int(g*255), int(b*255)))
    return colors

def build_cluster_cmap(clusters):
    """Build {cluster_id: hex_color} dict for any number of clusters."""
    sorted_c = sorted(clusters)
    colors = _generate_colors(len(sorted_c))
    return {c: colors[i] for i, c in enumerate(sorted_c)}

CLUSTER_CMAP = {}

def cluster_color(c, alpha=1.0):
    h = CLUSTER_CMAP.get(int(c), '#888888').lstrip('#')
    r, g, b = [int(h[i:i+2], 16) / 255 for i in (0, 2, 4)]
    return (r, g, b, alpha)

BG, AX_BG, TICK = '#0F0F0F', '#1A1A1A', '#AAAAAA'
PATTERN_C = {'unpatterned': '#95A5A6', 'ridged': '#E74C3C', 'whorl': '#9B59B6', 'dorsal': '#3498DB', 'back': '#27AE60', 'front': '#E67E22'}
DONOR_C = {'donor1': '#FF6B6B', 'donor2': '#4ECDC4', 'donor3': '#F39C12', 'donor4': '#96CEB4', 'donor5': '#FFEAA7'}

print('Resolutions: ' + str(LEIDEN_RESOLUTIONS))
print('Active resolution: ' + str(ACTIVE_RES))
print('Target features: ' + str(len(TARGET_FEATURES)))

## A4-aware figure export (run once)

`save_fig(fig, name)` writes an Illustrator-editable **SVG** (vector) and a
high-DPI **PNG** sidecar at the figure's **native resolution**. SVG is vector,
so you scale it freely in Illustrator without quality loss.

If you need exact physical sizing for an A4 layout, pass `mm=PANEL_FULL_MM`
(180x130mm), `PANEL_HALF_MM` (86x70mm), `A4_PORTRAIT_MM`, or `A4_LANDSCAPE_MM`.

Requires `kaleido` (`pip install -U kaleido`).


In [ ]:
import os

# ======================================================================
# A4-aware Plotly figure export (matches notebooks 03 + 04)
#
# Default = native resolution: save_fig(fig, "name") uses the figure's
# own width/height. SVG is vector, so you scale it freely in Illustrator
# without quality loss. Pass mm=PRESET only when you need exact physical
# sizing for an A4 layout.
# ======================================================================
MM_PER_INCH = 25.4
SVG_DPI = 72            # Illustrator SVG import dpi
PNG_DPI = 300           # sidecar raster preview

def mm_to_px(mm, dpi=SVG_DPI):
    return int(round(mm * dpi / MM_PER_INCH))

# Physical-size presets (max footprint = A4)
A4_PORTRAIT_MM  = (210, 297)
A4_LANDSCAPE_MM = (297, 210)
PANEL_FULL_MM   = (180, 130)   # full-width manuscript panel
PANEL_HALF_MM   = (86, 70)     # half-width / single-column panel

FIG_OUT_DIR = 'figures'


def save_fig(fig, name, *, mm=None, width_px=None, height_px=None,
             out_dir=None, formats=("svg", "png")):
    """Save a Plotly figure as SVG (vector, Illustrator-editable) + high-dpi PNG.

    Sizing:
        mm=(w, h)            -> sized to those mm at 72 dpi (explicit physical size).
        width_px / height_px -> raw pixel size.
        neither              -> NATIVE resolution (uses fig.layout.width / height).

    PNG is rendered at PNG_DPI / SVG_DPI (~4.17x) for crisp previews.
    Requires kaleido:  pip install -U kaleido
    If the static image backend (kaleido) is missing, writes an interactive HTML instead.
    """
    out_dir = out_dir if out_dir is not None else FIG_OUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    if mm is not None:
        width_px = mm_to_px(mm[0])
        height_px = mm_to_px(mm[1])
    png_scale = PNG_DPI / SVG_DPI

    paths = []
    try:
        for fmt in formats:
            path = os.path.join(out_dir, f"{name}.{fmt}")
            kw = {}
            if width_px:  kw["width"] = width_px
            if height_px: kw["height"] = height_px
            if fmt == "png":
                kw["scale"] = png_scale
            fig.write_image(path, format=fmt, **kw)
            paths.append(path)
            print(f"  -> {path}")
    except Exception as e:
        path = os.path.join(out_dir, f"{name}.html")
        fig.write_html(path, include_plotlyjs="cdn")
        print(f"  [no static export: {type(e).__name__}] -> {path}")
        return [path]
    return paths


## Metadata Map

Biological annotations for each stack: donor, developmental timepoint, tissue site, skin pattern.

In [ ]:
META_MAP = {
    '(14 wks) digit 5 unpatterned skin x40 stack-Smooth05_Resampled.tif':
        {'donor': 'donor1', 'timepoint': '14wk', 'tissue': 'volar', 'pattern': 'unpatterned'},
    '(13 wks) D3L ventral ridges x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'volar', 'pattern': 'ridged'},
    '(13 wks) D3L Ventral unpatterned skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'volar', 'pattern': 'unpatterned'},
    '(13 wks) D3L Dorsal skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
    '(13 wks) Back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'back', 'pattern': 'back'},
    '(13 wks) Front skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'front', 'pattern': 'front'},
    '(14 wks) ventral whorl x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'volar', 'pattern': 'whorl'},
    '(14 wks) back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'back', 'pattern': 'back'},
    '(14 wks) D3L dorsal x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
    '(12 wks) D2L ventral tip x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'volar', 'pattern': 'ridged'},
    '(12 wks) back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'back', 'pattern': 'back'},
    '(12 wks) front skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'front', 'pattern': 'front'},
    '(12 wks) D3L dorsal x40 stack-Smooth05_Resampled.tif':
        {'donor': 'donor5', 'timepoint': '12wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
}
print(str(len(META_MAP)) + ' stacks annotated')

## Load + Annotate

In [ ]:
df_raw = pd.read_csv(FEATURES_CSV)
for key in ['donor', 'timepoint', 'tissue', 'pattern']:
    df_raw[key] = df_raw['stack_name'].map(
        lambda s, k=key: META_MAP.get(s, {}).get(k, 'unknown')
    )
print('Loaded ' + str(len(df_raw)) + ' cells from ' + str(df_raw['stack_name'].nunique()) + ' stacks')

# Check for unmapped stacks
unmapped = df_raw[df_raw['donor'] == 'unknown']['stack_name'].unique()
if len(unmapped):
    print('WARNING: ' + str(len(unmapped)) + ' unmapped stacks:')
    for s in unmapped:
        print('  ' + s)

# Filter to available features
avail_feat = [f for f in TARGET_FEATURES if f in df_raw.columns]
missing_feat = [f for f in TARGET_FEATURES if f not in df_raw.columns]
if missing_feat:
    print('Missing features (' + str(len(missing_feat)) + '): ' + str(missing_feat))

meta_cols = [c for c in ['donor', 'timepoint', 'tissue', 'pattern', 'stack_name',
                          'label', 'centroid-0', 'centroid-1', 'centroid-2',
                          'qc_score', 'z_position',
                          'major_axis_z', 'major_axis_y', 'major_axis_x']
             if c in df_raw.columns and c not in avail_feat]

df = df_raw[avail_feat + meta_cols].dropna(subset=avail_feat).copy().reset_index(drop=True)
print('After dropna: ' + str(len(df)) + ' cells')
print('Available features: ' + str(len(avail_feat)) + '/' + str(len(TARGET_FEATURES)))

## Preprocessing + PCA

In [ ]:
def preprocess(X, features, winsorize_clip=5.0):
    """Auto log1p skewed features, RobustScaler, winsorize."""
    X = X.copy()
    log_applied = []
    for i, feat in enumerate(features):
        col = X[:, i]
        if col.min() >= 0 and abs(skew(col)) > 1.0:
            X[:, i] = np.log1p(col)
            log_applied.append(feat)
    X = RobustScaler().fit_transform(X)
    if winsorize_clip is not None:
        X = np.clip(X, -winsorize_clip, winsorize_clip)
    print('  Log1p (' + str(len(log_applied)) + '): ' + (', '.join(log_applied) or 'none'))
    return X

# Preprocess all target features
print('Target features:')
X_all = preprocess(df[avail_feat].to_numpy(dtype=float), avail_feat, WINSOR_CLIP)

if USE_RESIDUAL_PCA:
    size_avail = [f for f in SIZE_REGRESSORS if f in avail_feat]
    size_idx = [avail_feat.index(f) for f in size_avail]
    S = X_all[:, size_idx]
    size_pc1 = PCA(n_components=1, random_state=RANDOM_SEED).fit_transform(S).ravel()
    size_z = (size_pc1 - size_pc1.mean()) / (size_pc1.std() + 1e-12)
    non_size_idx = [i for i, f in enumerate(avail_feat) if f not in SIZE_REGRESSORS]
    non_size_feat = [avail_feat[i] for i in non_size_idx]
    X_input = X_all[:, non_size_idx].copy()
    for i in range(X_input.shape[1]):
        reg = LinearRegression().fit(size_z.reshape(-1, 1), X_input[:, i])
        X_input[:, i] -= reg.predict(size_z.reshape(-1, 1))
    avail_kf = non_size_feat
    print('Residual PCA: regressed size PC1 from ' + str(len(avail_kf)) + ' features')
else:
    X_input = X_all
    avail_kf = avail_feat
    print('Standard PCA on ' + str(len(avail_kf)) + ' features')

# PCA
n_pcs_actual = min(N_PCS, X_input.shape[1], X_input.shape[0])
pca = PCA(n_components=n_pcs_actual, random_state=RANDOM_SEED)
pcs = pca.fit_transform(X_input)
ev = pca.explained_variance_ratio_
cumvar = np.cumsum(ev)
pc_display = ['PC' + str(i + 1) for i in range(n_pcs_actual)]
print('\nExplained variance (' + str(n_pcs_actual) + ' PCs):')
for i, v in enumerate(ev):
    bar = '#' * int(v * 100)
    print('  PC' + str(i+1).rjust(2) + ': ' + '{:.3f}'.format(v) + '  cumul: ' + '{:.3f}'.format(ev[:i+1].sum()) + '  ' + bar)

# Loadings heatmap
loadings = pd.DataFrame(
    pca.components_.T, index=avail_kf,
    columns=['PC' + str(i+1) for i in range(n_pcs_actual)],
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
ax1.set_facecolor(AX_BG)
ax1.bar(range(1, n_pcs_actual + 1), ev, color='#6abf69')
ax1.set_xlabel('PC', color=TICK); ax1.set_ylabel('Variance explained', color=TICK)
ax1.set_title('Scree plot', color='white', fontsize=10)
ax1.tick_params(colors=TICK, labelsize=7)

ax2.set_facecolor(AX_BG)
show_pcs = min(10, n_pcs_actual)
im = ax2.imshow(loadings.values[:, :show_pcs], cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
ax2.set_xticks(range(show_pcs))
ax2.set_xticklabels(['PC' + str(i+1) for i in range(show_pcs)], fontsize=7, color=TICK)
ax2.set_yticks(range(len(avail_kf)))
ax2.set_yticklabels(avail_kf, fontsize=6, color=TICK)
ax2.set_title('Loadings', color='white', fontsize=10)
ax2.tick_params(colors=TICK)
plt.colorbar(im, ax=ax2, shrink=0.8)
plt.tight_layout(); plt.show()
print('PCA input shape: ' + str(X_input.shape) + ' -> PCs shape: ' + str(pcs.shape))

## Interactive PCA Loadings (Plotly)

Clustered heatmap with dendrogram and scree plot. Saves as interactive HTML.

In [ ]:
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import linkage

n_features, n_total_pcs = loadings.shape
show_pcs = min(show_pcs, n_total_pcs)
pc_display_use = pc_display[:show_pcs]
load_matrix = loadings.iloc[:, :show_pcs].to_numpy()

if load_matrix.shape[0] < 2:
    raise ValueError('Need at least 2 features/rows to cluster.')

# Build dendrogram
dendro_fig = ff.create_dendrogram(
    load_matrix,
    orientation='left',
    labels=avail_kf,
    linkagefun=lambda x: linkage(x, method='ward', metric='cosine'),
)

dendro_leaves = list(dendro_fig.layout.yaxis.ticktext)
row_order = [avail_kf.index(f) for f in dendro_leaves]

load_clustered = load_matrix[row_order, :]
feat_clustered = [avail_kf[i] for i in row_order]
n_leaves = len(feat_clustered)

all_raw_y, all_raw_x = [], []
for trace in dendro_fig.data:
    all_raw_y.extend(trace.y)
    all_raw_x.extend(trace.x)

raw_x_max = max(all_raw_x)

# Map the dendrogram's leaf positions onto heat-map rows 0..n-1 in the SAME order as
# feat_clustered (row 0 = dendro_leaves[0]). Plotly's left-oriented dendrogram places
# leaf k at tickvals[k] (negative, decreasing, in recent plotly versions), so the mapping
# goes through the tick values rather than assuming leaf 0 sits at the minimum y;
# the old (v - min_y) / 10 form drew the tree upside down relative to the rows.
dendro_tickvals = list(dendro_fig.layout.yaxis.tickvals)
leaf_step = dendro_tickvals[1] - dendro_tickvals[0]

def remap_y(y_vals):
    return tuple((v - dendro_tickvals[0]) / leaf_step for v in y_vals)

# Figure sizing
CELL_SIZE = 25
MARGIN_LEFT = 0
MARGIN_RIGHT = 80
MARGIN_TOP = 50
MARGIN_BOT = 120
SCREE_FRAC = 0.2
DENDRO_FRAC = 0.25
H_SPACING = 0.25

heatmap_height = n_leaves * CELL_SIZE
total_plot_area = heatmap_height / (1 - SCREE_FRAC)
fig_height = int(total_plot_area + MARGIN_TOP + MARGIN_BOT)
fig_width = max(200, 650)

fig = make_subplots(
    rows=2, cols=2,
    column_widths=[DENDRO_FRAC, 1 - DENDRO_FRAC],
    row_heights=[SCREE_FRAC, 1 - SCREE_FRAC],
    horizontal_spacing=H_SPACING,
    vertical_spacing=0.01,
    specs=[
        [None, {'type': 'xy'}],
        [{'type': 'xy'}, {'type': 'heatmap'}],
    ],
)

# Scree plot
bar_colors = [
    'rgba(78,121,167,' + '{:.2f}'.format(max(0.35, 0.95 - i * 0.06)) + ')'
    for i in range(show_pcs)
]
fig.add_trace(
    go.Bar(
        x=pc_display_use,
        y=ev[:show_pcs] * 100,
        name='% variance',
        marker=dict(color=bar_colors, line=dict(width=0)),
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=pc_display_use,
        y=cumvar[:show_pcs] * 100,
        name='cumulative',
        mode='lines+markers',
        line=dict(color='#E15759', width=2),
        marker=dict(size=5, color='#E15759', line=dict(color='white', width=1)),
    ),
    row=1, col=2,
)
fig.add_hline(
    y=80, row=1, col=2,
    line_dash='dot', line_color='#BBBBBB', line_width=1.2,
    annotation_text='80%',
    annotation_font=dict(size=9, color='#999999'),
    annotation_position='bottom right',
)

# Dendrogram
x_pad = raw_x_max * 0.05 if raw_x_max > 0 else 0.5
for trace in dendro_fig.data:
    fig.add_trace(
        go.Scatter(
            x=tuple(-v for v in trace.x),
            y=remap_y(trace.y),
            mode='lines',
            line=dict(
                color=trace.line.color if trace.line.color else '#888888',
                width=1.2,
            ),
            showlegend=False,
            hoverinfo='skip',
        ),
        row=2, col=1,
    )

# Heatmap
hover_text = [
    [
        feat_clustered[r] + ' | ' + pc_display_use[c] + ' | loading: ' + '{:+.3f}'.format(load_clustered[r, c])
        for c in range(show_pcs)
    ]
    for r in range(n_leaves)
]
fig.add_trace(
    go.Heatmap(
        z=load_clustered,
        x=pc_display_use,
        y=list(range(n_leaves)),
        colorscale=[
            [0.00, '#2166AC'],
            [0.25, '#92C5DE'],
            [0.48, '#F7F7F7'],
            [0.52, '#F7F7F7'],
            [0.75, '#F4A582'],
            [1.00, '#B2182B'],
        ],
        zmid=0, zmin=-0.5, zmax=0.5,
        text=hover_text, hoverinfo='text',
        xgap=2, ygap=2,
        colorbar=dict(
            title=dict(text='Component Loadings',
                       font=dict(size=10, color='#333333', family='Arial'),
                       side='right'),
            tickfont=dict(size=9, color='#333333', family='Arial'),
            tickvals=[-0.5, -0.25, 0, 0.25, 0.5],
            len=0.30, thickness=11,
            x=1.01, y=0.08, yanchor='bottom',
            outlinewidth=0.6, outlinecolor='#DDDDDD',
            ticks='outside', ticklen=3,
        ),
    ),
    row=2, col=2,
)

# Axis formatting
fig.update_xaxes(row=1, col=2, showgrid=False, showticklabels=False, ticks='', showline=False, zeroline=False)
fig.update_yaxes(row=1, col=2, title_text='% var. explained',
    title_font=dict(size=9, color='#555555'), tickfont=dict(size=8, color='#444444'),
    showgrid=True, gridcolor='#F0F0F0', linecolor='#DDDDDD',
    ticks='outside', ticklen=3, tickcolor='#DDDDDD', zeroline=False, range=[0, 108])
fig.update_xaxes(row=2, col=1, showticklabels=False, showgrid=False,
    zeroline=False, showline=False, ticks='',
    range=[-(raw_x_max + x_pad), x_pad], autorange=False)
fig.update_yaxes(row=2, col=1, showticklabels=False, showgrid=False,
    zeroline=False, showline=False, ticks='',
    range=[-0.5, n_leaves - 0.5], autorange=False)
fig.update_xaxes(row=2, col=2, tickangle=-42,
    tickfont=dict(size=8, color='#333333'), showgrid=False, linecolor='#DDDDDD', side='bottom')
fig.update_yaxes(row=2, col=2, tickvals=list(range(n_leaves)),
    ticktext=feat_clustered, tickfont=dict(size=8, color='#333333'),
    showgrid=False, linecolor='#DDDDDD', ticks='outside', ticklen=3, tickcolor='#DDDDDD',
    range=[-0.5, n_leaves - 0.5], autorange=False)

fig.update_layout(
    template='plotly_white', paper_bgcolor='white', plot_bgcolor='white',
    height=fig_height, width=fig_width,
    font=dict(family='Arial', size=25, color='#333333'),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=1.0, yanchor='bottom',
        bgcolor='rgba(255,255,255,0)', borderwidth=0, font=dict(size=9, color='#333333')),
    margin=dict(l=MARGIN_LEFT, r=MARGIN_RIGHT, t=MARGIN_TOP, b=MARGIN_BOT),
)
fig.show()
save_fig(fig, "pca_loadings_dendrogram_heatmap")  # native resolution; scale in Illustrator
out_path = os.path.join(OUTPUT_DIR, 'pca_loadings_stacked_layout_compact.html')
fig.write_html(out_path)
print('Saved ' + out_path)

## Multi-Resolution Leiden Clustering

In [ ]:
# Build KNN graph once
print('Building kNN graph...')
nn = NearestNeighbors(n_neighbors=KNN + 1, metric=METRIC).fit(pcs)
dists, idxs = nn.kneighbors(pcs)

edges, weights = [], []
for i in range(len(pcs)):
    for j, d in zip(idxs[i, 1:], dists[i, 1:]):
        edges.append((i, int(j)))
        weights.append(float(np.exp(-d)))

G = ig.Graph(n=len(pcs), edges=edges, directed=False)
G.es['weight'] = weights
G = G.simplify(combine_edges='mean')
w = np.array(G.es['weight'])
print('Graph  : ' + str(G.vcount()) + ' nodes / ' + str(G.ecount()) + ' edges')
print('Weights: [' + '{:.3f}'.format(w.min()) + ', ' + '{:.3f}'.format(w.max()) + ']')

# Run Leiden at each resolution
leiden_results = {}
for res in LEIDEN_RESOLUTIONS:
    part = leidenalg.find_partition(
        G, leidenalg.RBConfigurationVertexPartition,
        weights='weight', resolution_parameter=res, seed=RANDOM_SEED,
    )
    col_name = 'leiden_r' + str(res)
    df[col_name] = part.membership
    n_clust = len(set(part.membership))
    leiden_results[res] = {
        'col': col_name,
        'n_clusters': n_clust,
        'clusters': sorted(set(part.membership)),
    }
    print('  res=' + str(res).ljust(5) + ' -> ' + str(n_clust).rjust(3) + ' clusters')

# Set active resolution for downstream cells
active_col = 'leiden_r' + str(ACTIVE_RES)
assert active_col in df.columns, 'ACTIVE_RES=' + str(ACTIVE_RES) + ' not in LEIDEN_RESOLUTIONS'
df['leiden'] = df[active_col]
clusters = sorted(df['leiden'].unique())

# Build color map dynamically
all_cluster_ids = set()
for info in leiden_results.values():
    all_cluster_ids.update(info['clusters'])
CLUSTER_CMAP = build_cluster_cmap(all_cluster_ids)
cc = {c: CLUSTER_CMAP.get(c, '#888888') for c in clusters}

for res, info in leiden_results.items():
    info['cc'] = {c: CLUSTER_CMAP.get(c, '#888888') for c in info['clusters']}

print('\nActive resolution: ' + str(ACTIVE_RES) + ' (' + str(len(clusters)) + ' clusters)')
print(df['leiden'].value_counts().sort_index())

## UMAP Embedding

In [ ]:
# UMAP 2D
print('Running 2D UMAP...')
umap_2d = UMAP(
    n_components=2, n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_MIN_DIST,
    metric=METRIC, random_state=RANDOM_SEED,
).fit_transform(pcs)
df['umap_1'] = umap_2d[:, 0]
df['umap_2'] = umap_2d[:, 1]
print('  2D UMAP done')

print('\nAll embeddings computed for ' + str(len(df)) + ' cells')

## Sample Representative Cells

In [ ]:
# Build mesh lookup
mesh_lookup = {}
for sname in df['stack_name'].unique():
    mesh_dir = os.path.join(MESHES_ROOT, sname)
    if os.path.isdir(mesh_dir):
        for obj_file in glob.glob(os.path.join(mesh_dir, 'cell_*.obj')):
            fname = os.path.basename(obj_file)
            try:
                lbl = int(fname.replace('cell_', '').replace('.obj', ''))
                mesh_lookup[(sname, lbl)] = obj_file
            except ValueError:
                pass

manifests = glob.glob(os.path.join(MESHES_ROOT, '*', 'mesh_manifest.csv'))
for mpath in manifests:
    sname = Path(mpath).parent.name
    try:
        mdf = pd.read_csv(mpath)
        for _, row in mdf.iterrows():
            lbl = int(row.get('label', 0))
            obj_path = str(row.get('mesh_path', ''))
            if obj_path and not os.path.isabs(obj_path):
                obj_path = os.path.join(WORKSPACE_DIR, obj_path)
            if lbl and obj_path and os.path.exists(obj_path):
                mesh_lookup[(sname, lbl)] = obj_path
    except Exception:
        pass

n_with_mesh = sum(1 for _, row in df.iterrows()
    if (row['stack_name'], int(row['label'])) in mesh_lookup)
print('Mesh lookup: ' + str(len(mesh_lookup)) + ' total meshes')
print('Cells with meshes: ' + str(n_with_mesh) + '/' + str(len(df)))

def select_representatives(pcs, df, mesh_lookup, cluster_col='leiden', n_reps=10):
    """Select N cells closest to cluster median in PCA space."""
    rep_rows = []
    clusters_list = sorted(df[cluster_col].unique())
    for c in clusters_list:
        mask = df[cluster_col].values == c
        idx_arr = np.where(mask)[0]
        pcs_c = pcs[mask]
        median = np.median(pcs_c, axis=0)
        dists_c = np.linalg.norm(pcs_c - median, axis=1)
        n_pick = min(n_reps, len(idx_arr))
        closest = idx_arr[np.argsort(dists_c)[:n_pick]]
        for rank, ri in enumerate(closest):
            row = df.iloc[ri]
            sname = row['stack_name']
            lbl = int(row['label'])
            mpath = mesh_lookup.get((sname, lbl), None)
            local_idx = np.where(idx_arr == ri)[0][0]
            rep_rows.append({
                'cluster': int(c),
                'role': 'rep_' + str(rank),
                'rank': rank,
                'label': lbl,
                'stack_name': sname,
                'mesh_path': mpath if mpath else '',
                'mesh_exists': mpath is not None and os.path.exists(str(mpath)),
                'dist_to_median': float(dists_c[local_idx]),
            })
    return pd.DataFrame(rep_rows)

# Sample representatives for all resolutions
for res, info in leiden_results.items():
    col = info['col']
    res_dir = os.path.join(OUTPUT_DIR, 'leiden_res_' + str(res))
    os.makedirs(res_dir, exist_ok=True)
    res_rep_df = select_representatives(pcs, df, mesh_lookup, cluster_col=col, n_reps=N_REPS)
    res_rep_df.to_csv(os.path.join(res_dir, 'representative_cells.csv'), index=False)
    n_found = res_rep_df['mesh_exists'].sum()
    print('res=' + str(res) + ': ' + str(info['n_clusters']) + ' clusters, '
          + str(len(res_rep_df)) + ' reps (' + str(n_found) + ' with meshes)')

## Save Results

Save `morphospace_results.pkl` as the bridge to downstream notebooks.

In [ ]:
results = {
    'df': df,
    'pcs': pcs,
    'pca_loadings': loadings,
    'pca_variance': ev,
    'pca_cumvar': cumvar,
    'avail_kf': avail_kf,
    'leiden_results': leiden_results,
    'config': {
        'TARGET_FEATURES': TARGET_FEATURES,
        'SIZE_REGRESSORS': SIZE_REGRESSORS,
        'N_PCS': N_PCS,
        'KNN': KNN,
        'LEIDEN_RESOLUTIONS': LEIDEN_RESOLUTIONS,
        'ACTIVE_RES': ACTIVE_RES,
        'UMAP_NEIGHBORS': UMAP_NEIGHBORS,
        'UMAP_MIN_DIST': UMAP_MIN_DIST,
        'METRIC': METRIC,
        'RANDOM_SEED': RANDOM_SEED,
        'WINSOR_CLIP': WINSOR_CLIP,
        'USE_RESIDUAL_PCA': USE_RESIDUAL_PCA,
        'N_REPS': N_REPS,
        'MESHES_ROOT': MESHES_ROOT,
        'OUTPUT_DIR': OUTPUT_DIR,
        'WORKSPACE_DIR': WORKSPACE_DIR,
    },
    'CLUSTER_CMAP': CLUSTER_CMAP,
    'META_MAP': META_MAP,
    'mesh_lookup': mesh_lookup,
    'BG': BG,
    'AX_BG': AX_BG,
    'TICK': TICK,
    'PATTERN_C': PATTERN_C,
    'DONOR_C': DONOR_C,
}

pkl_path = os.path.join(OUTPUT_DIR, 'morphospace_results.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(results, f)
print('Saved: ' + pkl_path)

# CSV backup
csv_path = os.path.join(OUTPUT_DIR, 'all_features_multi_leiden.csv')
df.to_csv(csv_path, index=False)
print('Saved: ' + csv_path)

# Also export per-resolution CSVs + colormap JSONs
for res, info in leiden_results.items():
    col = info['col']
    res_dir = os.path.join(OUTPUT_DIR, 'leiden_res_' + str(res))
    os.makedirs(res_dir, exist_ok=True)
    export_df = df.copy()
    export_df['leiden'] = export_df[col]
    export_df.to_csv(os.path.join(res_dir, 'features_with_clusters.csv'), index=False)
    # Cluster shape stats
    stats_rows = []
    for c in info['clusters']:
        sub = df[df[col] == c]
        row = {'cluster': c, 'n': len(sub)}
        for f in avail_kf:
            row[f + '_mean'] = sub[f].mean() if f in sub.columns else np.nan
            row[f + '_std'] = sub[f].std() if f in sub.columns else np.nan
        stats_rows.append(row)
    pd.DataFrame(stats_rows).to_csv(
        os.path.join(res_dir, 'cluster_shape_stats.csv'), index=False)
    # Napari colormap JSON
    napari_cmaps = {}
    for stack in df['stack_name'].unique():
        sub = df[df['stack_name'] == stack]
        stack_cmap = {}
        for _, r in sub.iterrows():
            lbl_str = str(int(r['label']))
            cluster_id = int(r[col])
            stack_cmap[lbl_str] = CLUSTER_CMAP.get(cluster_id, '#888888')
        napari_cmaps[stack] = stack_cmap
    with open(os.path.join(res_dir, 'napari_cluster_colormaps.json'), 'w') as f:
        json.dump(napari_cmaps, f, indent=2)

print('\nAll resolutions exported to: ' + OUTPUT_DIR)